# Initialization and imports

In [1]:
#Celda exclusiva colab:
#"""
#Instalar dependencias
!pip install transformers adapters langid scikit-optimize mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 29.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.0/247.0 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 681.8/681.8 kB 45.1 MB/s eta 0:00:00
   ━

In [2]:
import os
os.chdir('/content')
!rm -rf VRID_language_proyect

!git clone --branch dev_Alonso --single-branch https://github.com/jitalo333/VRID_language_proyect
#Moverse a repositorio
os.chdir('/content/VRID_language_proyect/BERT')
#Montar drive
from google.colab import drive
drive.mount('/content/drive')
#"""

Cloning into 'VRID_language_proyect'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 123 (delta 48), reused 112 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (123/123), 1.30 MiB | 11.47 MiB/s, done.
Resolving deltas: 100% (48/48), done.
Mounted at /content/drive


In [3]:
import pandas as pd
import os
from preprocess import preprocess_record
from encoder import embed_texts, prepare_data
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from transformers import MarianMTModel, MarianTokenizer
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM
from translate_v2 import _translate_block_bart, _translate_block_Helsinki, _translate_block_nllb, translator

#Train model
import ast
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter
from ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, eval_model, safe_log_metric
from ML_pipeline_skp import mlflow_ckeckpoint

# 1) Preprocesamiento de los datos

In [4]:
# 1) Cargar datos
path = "/content/drive/MyDrive/VRID_NLP/code/VRID_proyect/"
filePATH = os.path.join(path, "data_concat_new.xlsx")
df = pd.read_excel(filePATH,
                   usecols=["Código VRID", "Título", "Resumen", "Keywords", "Interdisciplinario", "Transdisciplinario"]) \
       .fillna("")
# 2) Preprocesar los datos
cols = ["Título", "Resumen", "Keywords"]
df[cols] = df[cols].applymap(lambda x: "" if str(x).strip().upper() == "DESCONOCIDO" else str(x).strip())
df["text_for_embedding"] = df.apply(
    lambda r: preprocess_record(r["Título"], r["Resumen"], r["Keywords"]),
    axis=1
)
#df.to_excel("data_preprocessed.xlsx", index=False)

/tmp/ipython-input-1333657409.py:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[cols] = df[cols].applymap(lambda x: "" if str(x).strip().upper() == "DESCONOCIDO" else str(x).strip())


In [5]:
import langid
# Detección y traducción
def detect_and_translate(text):
    lang, _ = langid.classify(text)
    if lang == 'es':
        return 1
    else:
      return 0

df['idioma'] =df.apply(lambda r: detect_and_translate(r["text_for_embedding"]), axis=1)
df.to_excel("data_preprocess_lang_new.xlsx", index=False)

# 2) Traducción del texto

## BART

In [ ]:
df = df.iloc[0:5]

model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)
_translate_block_function = _translate_block_bart

trans = translator(model, tokenizer, _translate_block_function)
# Aplicar traducción a la columna 'text_for_embedding'
df["text_for_embedding_translated"] = df.apply(
    lambda r: trans.detect_and_translate(r.get("text_for_embedding", "")),
    axis=1
)
#df.to_excel("data_translated.xlsx", index=False)

In [ ]:
df["text_for_embedding_translated"]

## NLLB

In [ ]:
df = df.iloc[0:5]
#model_name = "facebook/nllb-200-3.3B"  # O "facebook/nllb-200-distilled-600M" para algo más liviano
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = NllbTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
_translate_block_function = _translate_block_nllb

trans = translator(model, tokenizer, _translate_block_function)
# Aplicar traducción a la columna 'text_for_embedding'
df["text_for_embedding_translated"] = df.apply(
    lambda r: trans.detect_and_translate(r.get("text_for_embedding", "")),
    axis=1
)
#df.to_excel("data_translated.xlsx", index=False)

In [ ]:
df["text_for_embedding_translated"]

## Text splitter

In [47]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from transformers import MarianMTModel, MarianTokenizer
import torch

class translator():
    def __init__(self, model, tokenizer, max_input_tokens=512):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        # Cargar modelo y tokenizer
        self.tokenizer = tokenizer
        self.model = model.to(self.device)
        self.max_input_tokens = max_input_tokens

    def split_text(self, text_to_split):
      # Splitter basado en el tokenizador de Helsinki (cuenta tokens reales)
      text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
          tokenizer=self.tokenizer,
          chunk_size=self.max_input_tokens,       # el encoder de Marian suele aceptar hasta ~512 tokens
          chunk_overlap=0,
          separators=["\n\n", ".", ",", " "]
      )

      texts = text_splitter.create_documents([text_to_split])
      return texts

    def translate_esp_en(self, text_to_split):
      #Split text
      texts = self.split_text(text_to_split)
      # Translate
      translated_chunks = []
      for chunk in texts:
          encoded = self.tokenizer(chunk.page_content, return_tensors="pt", truncation=True, max_length=512).to(self.device)
          with torch.inference_mode():
              out_ids = self.model.generate(
                  **encoded,
                  num_beams=4,
                  max_new_tokens=self.max_input_tokens,   # evita salidas cortas
                  no_repeat_ngram_size=3,
                  early_stopping=True
              )
          translated = self.tokenizer.batch_decode(out_ids, skip_special_tokens=True)[0]
          translated_chunks.append(translated)
      # Traducción final unida
      final_translation = "\n\n".join(translated_chunks)
      return final_translation


text_to_split = """
Escribe algo.
""".lower()

model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
translator = translator(model, tokenizer)
final_translation=translator.translate_esp_en(text_to_split)
print(final_translation)

Write something down.


# 3) Embedding text

In [ ]:
path = "/content/drive/MyDrive/VRID_NLP/code/VRID_proyect/"
filePATH = os.path.join(path, "data_concatenada_translated.xlsx")
df = pd.read_excel(filePATH)
# 1) Preparar textos y etiquetas
texts, labels = prepare_data(df)

# 2) Calcular embeddings
# Parámetros modelo
BASE_MODEL = "allenai/specter2_base"
ADAPTER_NAME = "allenai/specter2"

emb_texts = embed_texts(texts, BASE_MODEL, ADAPTER_NAME)

# 3) Guardar embeddings y etiquetas
df_dataset = pd.DataFrame(columns=["Código VRID", "labels", "embedings"])
df_dataset["Código VRID"] = df.loc[labels.index, "Código VRID"]
df_dataset["labels"] = labels
df_dataset["embedings"] = emb_texts.tolist()
df_dataset.to_excel("dataset_embed_translated.xlsx", index=False)

# 4) Train classifier

In [ ]:
path = "/content/drive/MyDrive/VRID_NLP/code/VRID_proyect/"
filePATH = os.path.join(path, "dataset_embed_translated.xlsx")
df = pd.read_excel(filePATH)
df['embedings'] = df['embedings'].apply(lambda x: np.array(ast.literal_eval(x)))
df.head()


y = df['labels'].to_numpy()
X = df['embedings'].to_numpy()
X = np.vstack(X)

seed = 7
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=seed, stratify=y)

# 2. Elegir modelos a probar
model_keys = [
    #'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    #'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print(X_train.shape)
print(X_test.shape)
print("📊 Comienzo:", Counter(y_train))
print("📊 Comienzo:", Counter(y_test))

In [ ]:
# 4. Ejecutar entrenamiento, validación y test con tus funciones
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, n_iter=2, sample_weight_On = True)

print(results_val)
for model in models_dicc.values():
  results_test = eval_model(model, X_test, y_test)
  print(results_test)

experiment_name = 'test_functions'
mlflow_ckeckpoint(results_val, models_dicc, X_test, y_test, experiment_name)